In [ ]:
# Colab dependencies
!pip -q install -U transformers datasets accelerate tqdm


# Recursive LLM Degradation — Combined Evaluation

This notebook is the **evaluation-only** pipeline for the recursive generation experiment.

It combines the two evaluation parts assigned to this work:

1. **Domain-specific medical evaluation**
   - Disease entity retention
   - Chemical entity retention
   - Entity precision / recall / F1
   - Entity frequency
   - Reference-based entity mismatch rate

2. **Model / behavioral evaluation**
   - Answer length
   - 3-gram repetition rate
   - Conditional answer-only perplexity

The same fixed 1,000 PubMedQA labeled examples are used as the gold evaluation set for every generation.

**Important:** the reference-based entity mismatch rate is a mismatch-with-gold proxy. It is not a definitive factual hallucination detector.

In [ ]:
!pip uninstall -y sympy
!pip install --no-cache-dir --force-reinstall sympy==1.14.0

## 1. Environment and project paths

In [ ]:
import ast
import math
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Seed-123 prediction files are directly under /content in this Colab session.
PROJECT_DIR = Path("/content")
PREDICTION_DIR = PROJECT_DIR
EVALUATION_DIR = PROJECT_DIR / "evaluation"
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-3B"
NER_MODEL_NAME = "HFpf/pubmedbert-bc5cdr-ner"
GENERATIONS = ["G0", "G1", "G2", "G3"]
PREDICTION_COLUMN = "predicted_answer"

print("Prediction directory:", PREDICTION_DIR)
print("Evaluation directory:", EVALUATION_DIR)
print("Generations:", GENERATIONS)

## 2. Load the fixed 1,000-example gold evaluation set

In [ ]:
labeled_data = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled",
    split="train",
)

gold_df = pd.DataFrame(labeled_data)
gold_df = gold_df.rename(columns={"long_answer": "gold_answer"})
gold_df["pubid"] = gold_df["pubid"].astype(str)

required_gold_columns = {"pubid", "question", "context", "gold_answer"}
missing_gold = required_gold_columns - set(gold_df.columns)
if missing_gold:
    raise ValueError(f"Missing gold columns: {sorted(missing_gold)}")

if len(gold_df) != 1000:
    raise ValueError(f"Expected 1,000 labeled examples, found {len(gold_df)}.")

if gold_df["pubid"].duplicated().any():
    raise ValueError("Gold evaluation set contains duplicate pubids.")

print("Gold evaluation rows:", len(gold_df))
print("Unique pubids:", gold_df["pubid"].nunique())
display(gold_df[["pubid", "question", "gold_answer"]].head(3))

## 3. Load generation prediction CSVs

In [ ]:
prediction_tables = {}

for generation in GENERATIONS:
    path = PREDICTION_DIR / f"{generation.lower()}_predictions.csv"

    if not path.exists():
        raise FileNotFoundError(f"{generation}: missing {path}")

    pred_df = pd.read_csv(path)

    if PREDICTION_COLUMN not in pred_df.columns:
        raise ValueError(
            f"{generation}: expected column '{PREDICTION_COLUMN}', "
            f"found {pred_df.columns.tolist()}"
        )

    pred_df["pubid"] = pred_df["pubid"].astype(str)

    if pred_df["pubid"].duplicated().any():
        raise ValueError(f"{generation}: duplicate pubids in prediction CSV.")

    if len(pred_df) != 1000:
        raise ValueError(
            f"{generation}: expected 1,000 predictions, found {len(pred_df)}."
        )

    merged = gold_df[["pubid", "question", "context", "gold_answer"]].merge(
        pred_df[["pubid", PREDICTION_COLUMN]],
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    missing_predictions = merged[PREDICTION_COLUMN].isna().sum()
    if missing_predictions:
        raise ValueError(
            f"{generation}: {missing_predictions} gold examples have no prediction."
        )

    prediction_tables[generation] = merged
    print(f"{generation}: loaded {len(merged)} aligned predictions")

print("\nAvailable generations:", list(prediction_tables))


## 4. Medical NER setup

PubMedBERT BC5CDR is used **only for evaluation**, not for QLoRA training.

The model provides the Disease and Chemical categories used by the domain-specific evaluation.

In [ ]:
NER_DEVICE = 0 if torch.cuda.is_available() else -1

pubmed_ner_pipeline = pipeline(
    "token-classification",
    model=NER_MODEL_NAME,
    aggregation_strategy="simple",
    device=NER_DEVICE,
    stride=128,
)

print("NER model:", NER_MODEL_NAME)
print("NER device:", "cuda:0" if NER_DEVICE == 0 else "cpu")
print("Tokenizer max length:", pubmed_ner_pipeline.tokenizer.model_max_length)

## 5. NER and entity-matching helpers

In [ ]:
LABEL_MAP = {
    "LABEL_0": ("O", None),
    "LABEL_1": ("B", "Chemical"),
    "LABEL_2": ("B", "Disease"),
    "LABEL_3": ("I", "Disease"),
    "LABEL_4": ("I", "Chemical"),
}


def label_to_entity_type(label):
    # Some versions of the pipeline may already return the human-readable type.
    if label in {"Disease", "Chemical"}:
        return "B", label

    return LABEL_MAP.get(label, (None, None))


def merge_pubmed_predictions(text, predictions):
    """
    Convert PubMedBERT token-level predictions into entity spans.
    Works with either LABEL_* outputs or Disease/Chemical labels.
    """
    pieces = []

    for pred in predictions:
        label = pred.get("entity_group", pred.get("entity"))
        prefix, entity_type = label_to_entity_type(label)

        if entity_type is None:
            continue

        start = pred.get("start")
        end = pred.get("end")

        if start is None or end is None:
            continue

        pieces.append({
            "prefix": prefix,
            "type": entity_type,
            "start": int(start),
            "end": int(end),
            "score": float(pred.get("score", 0.0)),
        })

    pieces.sort(key=lambda x: (x["start"], x["end"]))

    merged = []

    for piece in pieces:
        if piece["prefix"] == "B":
            merged.append({
                "text": text[piece["start"]:piece["end"]],
                "type": piece["type"],
                "start": piece["start"],
                "end": piece["end"],
                "score": piece["score"],
            })
            continue

        if (
            merged
            and merged[-1]["type"] == piece["type"]
            and piece["start"] <= merged[-1]["end"] + 2
        ):
            merged[-1]["end"] = piece["end"]
            merged[-1]["text"] = text[
                merged[-1]["start"]:merged[-1]["end"]
            ]
            merged[-1]["score"] = (
                merged[-1]["score"] + piece["score"]
            ) / 2.0
        else:
            merged.append({
                "text": text[piece["start"]:piece["end"]],
                "type": piece["type"],
                "start": piece["start"],
                "end": piece["end"],
                "score": piece["score"],
            })

    return merged


def normalize_entity_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.replace("##", "")
    text = re.sub(r"\s+", " ", text).strip(" \t\n.,;:()[]{}\"'")
    return text.lower()


def deduplicate_entities(entities):
    cleaned = []
    seen = set()

    for entity in entities:
        normalized = normalize_entity_text(entity["text"])
        if not normalized:
            continue

        key = (
            entity["type"],
            entity["start"],
            entity["end"],
            normalized,
        )

        if key in seen:
            continue

        seen.add(key)

        cleaned.append({
            "text": entity["text"].strip(),
            "normalized_text": normalized,
            "type": entity["type"],
            "start": entity["start"],
            "end": entity["end"],
            "score": float(entity["score"]),
        })

    return cleaned


def run_ner(texts, batch_size=8, description="NER"):
    all_predictions = []

    texts = ["" if pd.isna(x) else str(x) for x in texts]

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc=description,
    ):
        batch = texts[start:start + batch_size]
        batch_predictions = pubmed_ner_pipeline(
            batch,
            batch_size=batch_size,
        )
        all_predictions.extend(batch_predictions)

    entities = [
        deduplicate_entities(
            merge_pubmed_predictions(text, predictions)
        )
        for text, predictions in zip(texts, all_predictions)
    ]

    return entities


def get_entity_set(entities, entity_type):
    return {
        entity["normalized_text"]
        for entity in entities
        if entity["type"] == entity_type
        and entity["normalized_text"]
    }


def calculate_sample_metrics(gold_entities, pred_entities, entity_type):
    gold_set = get_entity_set(gold_entities, entity_type)
    pred_set = get_entity_set(pred_entities, entity_type)

    matched = gold_set & pred_set

    gold_count = len(gold_set)
    pred_count = len(pred_set)
    matched_count = len(matched)

    recall = matched_count / gold_count if gold_count else np.nan
    precision = matched_count / pred_count if pred_count else np.nan

    if np.isfinite(precision) and np.isfinite(recall) and precision + recall:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = np.nan

    return {
        "gold_count": gold_count,
        "pred_count": pred_count,
        "matched_count": matched_count,
        "retention": recall,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def micro_metrics(df, prefix):
    gold_disease = df["gold_disease_count"].sum()
    pred_disease = df[f"{prefix}_disease_count"].sum()
    matched_disease = df[f"{prefix}_matched_disease_count"].sum()

    disease_precision = (
        matched_disease / pred_disease if pred_disease else 0.0
    )
    disease_recall = (
        matched_disease / gold_disease if gold_disease else 0.0
    )
    disease_f1 = (
        2 * disease_precision * disease_recall
        / (disease_precision + disease_recall)
        if disease_precision + disease_recall
        else 0.0
    )

    gold_chemical = df["gold_chemical_count"].sum()
    pred_chemical = df[f"{prefix}_chemical_count"].sum()
    matched_chemical = df[f"{prefix}_matched_chemical_count"].sum()

    chemical_precision = (
        matched_chemical / pred_chemical if pred_chemical else 0.0
    )
    chemical_recall = (
        matched_chemical / gold_chemical if gold_chemical else 0.0
    )
    chemical_f1 = (
        2 * chemical_precision * chemical_recall
        / (chemical_precision + chemical_recall)
        if chemical_precision + chemical_recall
        else 0.0
    )

    return {
        "disease_gold": gold_disease,
        "disease_predicted": pred_disease,
        "disease_matched": matched_disease,
        "disease_precision": disease_precision,
        "disease_recall": disease_recall,
        "disease_f1": disease_f1,
        "chemical_gold": gold_chemical,
        "chemical_predicted": pred_chemical,
        "chemical_matched": matched_chemical,
        "chemical_precision": chemical_precision,
        "chemical_recall": chemical_recall,
        "chemical_f1": chemical_f1,
    }

## 6. Compute gold medical entities once

In [ ]:
gold_entities_clean = run_ner(
    gold_df["gold_answer"].tolist(),
    batch_size=8,
    description="Gold NER",
)

print("Gold entity sets:", len(gold_entities_clean))

# Quick inspection
for i in range(min(3, len(gold_entities_clean))):
    print("=" * 70)
    print(f"Example {i + 1}")
    for entity in gold_entities_clean[i]:
        print(f"{entity['text']} -> {entity['type']}")

## 7. Behavioral metrics and fixed Qwen evaluator

In [ ]:
eval_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
eval_tokenizer.padding_side = "left"

if eval_tokenizer.pad_token is None:
    eval_tokenizer.pad_token = eval_tokenizer.eos_token

eval_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
eval_model.eval()

print("Fixed evaluator loaded:", MODEL_NAME)
print("Device:", eval_model.device)


def context_to_string(context):
    # CSV serialization can turn the original dict/list context into a string.
    if isinstance(context, str):
        raw = context.strip()
        if raw.startswith("{") or raw.startswith("["):
            try:
                context = ast.literal_eval(raw)
            except (ValueError, SyntaxError):
                pass

    if isinstance(context, dict):
        contexts = context.get("contexts")
        if isinstance(contexts, (list, tuple)):
            return " ".join(str(x) for x in contexts)
        if contexts is not None:
            return str(contexts)
        return str(context)

    if isinstance(context, (list, tuple)):
        return " ".join(str(x) for x in context)

    return str(context)


def build_prompt(row):
    return (
        f"### Question:\n{row['question']}\n\n"
        f"### Context:\n{context_to_string(row['context'])}\n\n"
        "### Long Answer:\n"
    )


def word_count(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())


def token_count(text):
    if not isinstance(text, str) or not text.strip():
        return 0

    return len(
        eval_tokenizer(
            text,
            add_special_tokens=False,
        )["input_ids"]
    )


def repetition_rate(text, n=3):
    if not isinstance(text, str) or not text.strip():
        return 0.0

    words = re.findall(r"\b\w+\b", text.lower())

    if len(words) < n:
        return 0.0

    ngrams = [
        tuple(words[i:i+n])
        for i in range(len(words) - n + 1)
    ]

    return 1.0 - (len(set(ngrams)) / len(ngrams))


def calculate_batch_perplexity(rows, answer_column):
    input_id_lists = []
    label_id_lists = []

    for row in rows:
        prompt_ids = eval_tokenizer(
            build_prompt(row),
            add_special_tokens=False,
            truncation=True,
            max_length=1024,
        )["input_ids"]

        answer_ids = eval_tokenizer(
            str(row[answer_column]),
            add_special_tokens=False,
        )["input_ids"]

        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids

        input_id_lists.append(input_ids)
        label_id_lists.append(labels)

    batch = eval_tokenizer.pad(
        [{"input_ids": ids} for ids in input_id_lists],
        padding=True,
        return_tensors="pt",
    )

    input_ids = batch["input_ids"].to(eval_model.device)
    attention_mask = batch["attention_mask"].to(eval_model.device)

    max_len = input_ids.shape[1]
    labels = torch.full(
        (len(label_id_lists), max_len),
        -100,
        dtype=torch.long,
    )

    for i, label_ids in enumerate(label_id_lists):
        pad_length = max_len - len(label_ids)
        labels[i, pad_length:] = torch.tensor(
            label_ids,
            dtype=torch.long,
        )

    labels = labels.to(eval_model.device)

    with torch.no_grad():
        logits = eval_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        ).logits

    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    loss_per_token = torch.nn.functional.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        reduction="none",
        ignore_index=-100,
    ).view(shift_labels.shape)

    perplexities = []

    for i in range(len(rows)):
        valid_tokens = shift_labels[i] != -100

        if not valid_tokens.any():
            perplexities.append(np.nan)
            continue

        mean_loss = loss_per_token[i][valid_tokens].mean().item()
        perplexities.append(math.exp(mean_loss))

    return perplexities

## 8. Reusable evaluator for G0 / G1 / G2 / G3

In [ ]:
def evaluate_generation(
    eval_df,
    prediction_column,
    generation_name,
    gold_entities,
    ner_batch_size=8,
    ppl_batch_size=4,
):
    """
    Evaluate one generation on the same fixed gold evaluation set.

    Returns:
        result_df   : per-sample metrics
        summary_df  : one-row generation summary
    """
    short_name = generation_name.lower()

    texts = (
        eval_df[prediction_column]
        .fillna("")
        .astype(str)
        .tolist()
    )

    # -----------------------------
    # A. Medical NER
    # -----------------------------
    pred_entities = run_ner(
        texts,
        batch_size=ner_batch_size,
        description=f"{generation_name} NER",
    )

    domain_rows = []

    for i, (gold_entities_i, pred_entities_i) in enumerate(
        zip(gold_entities, pred_entities)
    ):
        disease = calculate_sample_metrics(
            gold_entities_i,
            pred_entities_i,
            "Disease",
        )

        chemical = calculate_sample_metrics(
            gold_entities_i,
            pred_entities_i,
            "Chemical",
        )

        domain_rows.append({
            "pubid": eval_df.iloc[i]["pubid"],

            "gold_disease_count": disease["gold_count"],
            f"{short_name}_disease_count": disease["pred_count"],
            f"{short_name}_matched_disease_count": disease["matched_count"],
            f"{short_name}_disease_retention": disease["retention"],
            f"{short_name}_disease_precision": disease["precision"],
            f"{short_name}_disease_recall": disease["recall"],
            f"{short_name}_disease_f1": disease["f1"],

            "gold_chemical_count": chemical["gold_count"],
            f"{short_name}_chemical_count": chemical["pred_count"],
            f"{short_name}_matched_chemical_count": chemical["matched_count"],
            f"{short_name}_chemical_retention": chemical["retention"],
            f"{short_name}_chemical_precision": chemical["precision"],
            f"{short_name}_chemical_recall": chemical["recall"],
            f"{short_name}_chemical_f1": chemical["f1"],
        })

    result_df = pd.DataFrame(domain_rows)

    # -----------------------------
    # B. Behavioral metrics
    # -----------------------------
    behavioral = pd.DataFrame({
        "pubid": eval_df["pubid"],
        f"{short_name}_word_count": [word_count(x) for x in texts],
        f"{short_name}_token_count": [token_count(x) for x in texts],
        f"{short_name}_repetition_rate": [
            repetition_rate(x) for x in texts
        ],
    })

    result_df = result_df.merge(
        behavioral,
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    # -----------------------------
    # C. Conditional PPL
    # -----------------------------
    rows = eval_df.to_dict("records")
    perplexities = []

    for start in tqdm(
        range(0, len(rows), ppl_batch_size),
        desc=f"{generation_name} perplexity",
    ):
        batch_rows = rows[start:start + ppl_batch_size]
        perplexities.extend(
            calculate_batch_perplexity(
                batch_rows,
                prediction_column,
            )
        )

    result_df[f"{short_name}_perplexity"] = perplexities

    # -----------------------------
    # D. Reference-based mismatch
    # -----------------------------
    pred_disease_total = result_df[
        f"{short_name}_disease_count"
    ].sum()

    matched_disease_total = result_df[
        f"{short_name}_matched_disease_count"
    ].sum()

    pred_chemical_total = result_df[
        f"{short_name}_chemical_count"
    ].sum()

    matched_chemical_total = result_df[
        f"{short_name}_matched_chemical_count"
    ].sum()

    disease_reference_mismatch = (
        1.0 - matched_disease_total / pred_disease_total
        if pred_disease_total > 0
        else 0.0
    )

    chemical_reference_mismatch = (
        1.0 - matched_chemical_total / pred_chemical_total
        if pred_chemical_total > 0
        else 0.0
    )

    # -----------------------------
    # E. Summary
    # -----------------------------
    micro = micro_metrics(result_df, short_name)

    summary = {
        "generation": generation_name,

        "disease_retention": micro["disease_recall"],
        "disease_precision": micro["disease_precision"],
        "disease_recall": micro["disease_recall"],
        "disease_f1": micro["disease_f1"],
        "disease_reference_mismatch_rate": disease_reference_mismatch,

        "chemical_retention": micro["chemical_recall"],
        "chemical_precision": micro["chemical_precision"],
        "chemical_recall": micro["chemical_recall"],
        "chemical_f1": micro["chemical_f1"],
        "chemical_reference_mismatch_rate": chemical_reference_mismatch,

        "gold_disease_entities_per_answer":
            result_df["gold_disease_count"].mean(),
        "predicted_disease_entities_per_answer":
            result_df[f"{short_name}_disease_count"].mean(),

        "gold_chemical_entities_per_answer":
            result_df["gold_chemical_count"].mean(),
        "predicted_chemical_entities_per_answer":
            result_df[f"{short_name}_chemical_count"].mean(),

        "avg_words":
            result_df[f"{short_name}_word_count"].mean(),
        "avg_tokens":
            result_df[f"{short_name}_token_count"].mean(),
        "avg_repetition_rate":
            result_df[f"{short_name}_repetition_rate"].mean(),

        "mean_perplexity":
            result_df[f"{short_name}_perplexity"].mean(),
        "median_perplexity":
            result_df[f"{short_name}_perplexity"].median(),
    }

    summary_df = pd.DataFrame([summary])

    return result_df, summary_df

## 9. Run the combined evaluation

In [ ]:
all_generation_results = {}
all_generation_summaries = []

for generation, generation_df in prediction_tables.items():
    result_df, summary_df = evaluate_generation(
        eval_df=generation_df,
        prediction_column=PREDICTION_COLUMN,
        generation_name=generation,
        gold_entities=gold_entities_clean,
    )

    all_generation_results[generation] = result_df
    all_generation_summaries.append(summary_df)

    per_generation_path = (
        EVALUATION_DIR / f"{generation.lower()}_evaluation.csv"
    )
    result_df.to_csv(per_generation_path, index=False)

    print(f"\nSaved: {per_generation_path}")

combined_summary = pd.concat(
    all_generation_summaries,
    ignore_index=True,
)

display(combined_summary)

## 10. Final comparison table

This is the main table to use for the domain-specific + behavioral part of the study.

The lexical and semantic metrics from the partner's evaluation can be merged into this table later using the `generation` column.

In [ ]:
comparison_columns = [
    "generation",

    "disease_retention",
    "disease_precision",
    "disease_recall",
    "disease_f1",
    "disease_reference_mismatch_rate",

    "chemical_retention",
    "chemical_precision",
    "chemical_recall",
    "chemical_f1",
    "chemical_reference_mismatch_rate",

    "gold_disease_entities_per_answer",
    "predicted_disease_entities_per_answer",
    "gold_chemical_entities_per_answer",
    "predicted_chemical_entities_per_answer",

    "avg_words",
    "avg_tokens",
    "avg_repetition_rate",

    "mean_perplexity",
    "median_perplexity",
]

comparison_table = combined_summary[comparison_columns].copy()

display(comparison_table)

comparison_path = EVALUATION_DIR / "combined_domain_behavioral_summary.csv"
comparison_table.to_csv(comparison_path, index=False)

print("Saved:", comparison_path)

In [ ]:
# Optional: combine all per-sample results across generations.
# Each row corresponds to one pubid.

combined_per_sample = None

for generation, result_df in all_generation_results.items():
    prefix = generation.lower()

    columns_to_keep = [
        "pubid",
        "gold_disease_count",
        "gold_chemical_count",
        f"{prefix}_disease_count",
        f"{prefix}_matched_disease_count",
        f"{prefix}_disease_retention",
        f"{prefix}_disease_precision",
        f"{prefix}_disease_recall",
        f"{prefix}_disease_f1",
        f"{prefix}_chemical_count",
        f"{prefix}_matched_chemical_count",
        f"{prefix}_chemical_retention",
        f"{prefix}_chemical_precision",
        f"{prefix}_chemical_recall",
        f"{prefix}_chemical_f1",
        f"{prefix}_word_count",
        f"{prefix}_token_count",
        f"{prefix}_repetition_rate",
        f"{prefix}_perplexity",
    ]

    current = result_df[columns_to_keep].copy()

    if combined_per_sample is None:
        combined_per_sample = current
    else:
        drop_gold = [
            "gold_disease_count",
            "gold_chemical_count",
        ]
        current = current.drop(columns=drop_gold)
        combined_per_sample = combined_per_sample.merge(
            current,
            on="pubid",
            how="left",
            validate="one_to_one",
        )

per_sample_path = EVALUATION_DIR / "combined_domain_behavioral_per_sample.csv"
combined_per_sample.to_csv(per_sample_path, index=False)

print("Saved:", per_sample_path)
display(combined_per_sample.head(3))

In [ ]:
summary_path = Path("/content/evaluation/combined_domain_behavioral_summary.csv")

if summary_path.exists():
    summary = pd.read_csv(summary_path)
    display(summary.round(4))
else:
    print("Summary file not found. Run the notebook's final comparison-table cell first.")

In [ ]:
print("NER label mapping:")
print(LABEL_MAP)

print("\nSample gold entities:")
for e in gold_entities_clean[0]:
    print(e)

In [ ]:
from transformers import AutoConfig

ner_config = AutoConfig.from_pretrained("HFpf/pubmedbert-bc5cdr-ner")

print("id2label:")
print(ner_config.id2label)

print("\nlabel2id:")
print(ner_config.label2id)

# context-supported vs unsupported entity analysis

In [ ]:
import re
import unicodedata
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

def normalize_text_simple(text):
    text = unicodedata.normalize("NFKC", str(text)).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def entity_in_text(entity, text):
    entity_norm = normalize_text_simple(entity["text"])
    text_norm = normalize_text_simple(text)
    return bool(entity_norm) and entity_norm in text_norm

def context_support_metrics(contexts, pred_entities):
    supported = 0
    unsupported = 0

    for entity in pred_entities:
        if entity_in_text(entity, contexts):
            supported += 1
        else:
            unsupported += 1

    total = supported + unsupported

    return {
        "supported_count": supported,
        "unsupported_count": unsupported,
        "supported_rate": supported / total if total else np.nan,
        "unsupported_rate": unsupported / total if total else np.nan,
    }

In [ ]:
context_support_results = {}

for generation, generation_df in prediction_tables.items():
    texts = (
        generation_df["predicted_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    pred_entities = run_ner(
        texts,
        batch_size=8,
        description=f"{generation} context-support NER"
    )

    rows = []

    for i, entities in enumerate(pred_entities):
        context_text = context_to_string(
            generation_df.iloc[i]["context"]
        )

        metrics = context_support_metrics(
            context_text,
            entities
        )

        rows.append({
            "pubid": generation_df.iloc[i]["pubid"],
            "supported_entity_count": metrics["supported_count"],
            "unsupported_entity_count": metrics["unsupported_count"],
            "context_supported_rate": metrics["supported_rate"],
            "context_unsupported_rate": metrics["unsupported_rate"],
        })

    result = pd.DataFrame(rows)
    context_support_results[generation] = result

    path = EVALUATION_DIR / f"{generation.lower()}_context_support.csv"
    result.to_csv(path, index=False)

    print(f"{generation}: saved {path}")

In [ ]:
context_summary = []

for generation, df in context_support_results.items():
    context_summary.append({
        "generation": generation,
        "supported_entities_per_answer": df["supported_entity_count"].mean(),
        "unsupported_entities_per_answer": df["unsupported_entity_count"].mean(),
        "context_supported_rate": df["context_supported_rate"].mean(),
        "context_unsupported_rate": df["context_unsupported_rate"].mean(),
    })

context_summary = pd.DataFrame(context_summary)

display(context_summary.round(4))

# answer diversity

In [ ]:
from collections import Counter
import re
import numpy as np
import pandas as pd

def distinct_n(texts, n):
    total_ngrams = 0
    unique_ngrams = set()

    for text in texts:
        words = re.findall(r"\b\w+\b", str(text).lower())

        if len(words) < n:
            continue

        ngrams = [
            tuple(words[i:i+n])
            for i in range(len(words) - n + 1)
        ]

        total_ngrams += len(ngrams)
        unique_ngrams.update(ngrams)

    return len(unique_ngrams) / total_ngrams if total_ngrams else np.nan


diversity_rows = []

for generation, df in prediction_tables.items():
    answers = (
        df["predicted_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    diversity_rows.append({
        "generation": generation,
        "distinct_1": distinct_n(answers, 1),
        "distinct_2": distinct_n(answers, 2),
        "unique_answer_ratio": (
            len(set(answers)) / len(answers)
            if answers else np.nan
        ),
    })

diversity_summary = pd.DataFrame(diversity_rows)

display(diversity_summary.round(4))

diversity_summary.to_csv(
    EVALUATION_DIR / "diversity_summary.csv",
    index=False
)

## Evaluation notes

- The fixed gold set is the same 1,000 PubMedQA labeled examples for every generation.
- Prediction CSVs are aligned by `pubid`, not by row order.
- PubMedBERT BC5CDR is used only for Disease/Chemical entity extraction during evaluation.
- Disease/Chemical retention is equivalent to entity recall under the matching rule used here.
- Reference-based entity mismatch is a gold-reference mismatch proxy, not a definitive factual hallucination rate.
- Perplexity is conditional answer-only perplexity under the fixed `Qwen/Qwen2.5-0.5B` evaluator.
- The evaluation notebook does not train or generate answers; it only consumes the prediction CSVs produced by the recursive-generation pipeline.

# lexical/semantic metrics: BLEU, ROUGE-L, METEOR, chrF++, BERTScore, cosine similarity

# it start from here

In [ ]:
!pip -q install nltk sacrebleu rouge-score bert-score sentence-transformers

# Imports

In [ ]:
import nltk
import sacrebleu
import pandas as pd
import numpy as np
import torch

from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

nltk.download("wordnet")
nltk.download("omw-1.4")

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Lexical metric function

In [ ]:
smoothing = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def calculate_lexical_metrics(row):
    reference = str(row["long_answer"])
    prediction = str(row["predicted_answer"])

    reference_tokens = reference.split()
    prediction_tokens = prediction.split()

    bleu_1 = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        weights=(1.0, 0, 0, 0),
        smoothing_function=smoothing,
    )

    bleu_2 = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        weights=(0.5, 0.5, 0, 0),
        smoothing_function=smoothing,
    )

    bleu_3 = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        weights=(1/3, 1/3, 1/3, 0),
        smoothing_function=smoothing,
    )

    bleu_4 = sentence_bleu(
        [reference_tokens],
        prediction_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smoothing,
    )

    rouge_l = rouge.score(reference, prediction)["rougeL"].fmeasure

    meteor = meteor_score(
        [reference_tokens],
        prediction_tokens
    )

    chrf_pp = sacrebleu.sentence_chrf(
        prediction,
        [reference],
        word_order=2
    ).score

    return pd.Series({
        "bleu_1": bleu_1,
        "bleu_2": bleu_2,
        "bleu_3": bleu_3,
        "bleu_4": bleu_4,
        "rouge_l": rouge_l,
        "meteor": meteor,
        "chrf_pp": chrf_pp,
    })

# Run lexical evaluation for G0–G3

In [ ]:
# Make our current evaluation tables compatible with the partner's metric code.

for generation, df in prediction_tables.items():
    df["long_answer"] = df["gold_answer"]

print("Column compatibility fixed.")

for generation, df in prediction_tables.items():
    print(
        generation,
        "->",
        "long_answer" in df.columns,
        "predicted_answer" in df.columns
    )

In [ ]:
lexical_results = {}
lexical_summaries = []

for generation, generation_df in prediction_tables.items():

    print(f"\n===== {generation} =====")

    df = generation_df.copy()

    lexical_metrics = df.apply(
        calculate_lexical_metrics,
        axis=1
    )

    df = pd.concat(
        [df.reset_index(drop=True),
         lexical_metrics.reset_index(drop=True)],
        axis=1
    )

    lexical_results[generation] = df

    summary = {
        "generation": generation,
        "bleu_1": df["bleu_1"].mean(),
        "bleu_2": df["bleu_2"].mean(),
        "bleu_3": df["bleu_3"].mean(),
        "bleu_4": df["bleu_4"].mean(),
        "rouge_l": df["rouge_l"].mean(),
        "meteor": df["meteor"].mean(),
        "chrf_pp": df["chrf_pp"].mean(),
    }

    lexical_summaries.append(summary)

    print(pd.DataFrame([summary]).round(4))

# Save lexical results

In [ ]:
lexical_summary = pd.DataFrame(lexical_summaries)

display(
    lexical_summary.round(4)
)

lexical_summary.to_csv(
    EVALUATION_DIR / "lexical_summary_g0_g3.csv",
    index=False
)

for generation, df in lexical_results.items():
    df.to_csv(
        EVALUATION_DIR / f"{generation.lower()}_lexical_metrics_per_row.csv",
        index=False
    )

# BERTScore

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print("BERTScore device:", device)

# Evaluate BERTScore for G0–G3

In [ ]:
bertscore_results = {}
bertscore_summaries = []

for generation, generation_df in prediction_tables.items():

    print(f"\n===== BERTScore: {generation} =====")

    references = (
        generation_df["long_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    predictions = (
        generation_df["predicted_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    P, R, F1 = score(
        predictions,
        references,
        lang="en",
        device=device,
        batch_size=16,
        verbose=True,
    )

    result = pd.DataFrame({
        "pubid": generation_df["pubid"],
        "bertscore_precision": P.cpu().numpy(),
        "bertscore_recall": R.cpu().numpy(),
        "bertscore_f1": F1.cpu().numpy(),
    })

    bertscore_results[generation] = result

    summary = {
        "generation": generation,
        "bertscore_precision": result["bertscore_precision"].mean(),
        "bertscore_recall": result["bertscore_recall"].mean(),
        "bertscore_f1": result["bertscore_f1"].mean(),
    }

    bertscore_summaries.append(summary)

    print(pd.DataFrame([summary]).round(4))

In [ ]:
bertscore_summary = pd.DataFrame(bertscore_summaries)

display(
    bertscore_summary.round(4)
)

bertscore_summary.to_csv(
    EVALUATION_DIR / "bertscore_summary_g0_g3.csv",
    index=False
)

for generation, df in bertscore_results.items():
    df.to_csv(
        EVALUATION_DIR / f"{generation.lower()}_bertscore.csv",
        index=False
    )

# Cosine similarity

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

print("Embedding model loaded.")

In [ ]:
cosine_results = {}
cosine_summaries = []

for generation, generation_df in prediction_tables.items():

    print(f"\n===== Cosine similarity: {generation} =====")

    references = (
        generation_df["long_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    predictions = (
        generation_df["predicted_answer"]
        .fillna("")
        .astype(str)
        .tolist()
    )

    reference_embeddings = embedding_model.encode(
        references,
        batch_size=32,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    prediction_embeddings = embedding_model.encode(
        predictions,
        batch_size=32,
        convert_to_tensor=True,
        show_progress_bar=True,
    )

    scores = []

    for i in range(len(generation_df)):
        similarity = cos_sim(
            reference_embeddings[i],
            prediction_embeddings[i]
        ).item()

        scores.append(similarity)

    result = pd.DataFrame({
        "pubid": generation_df["pubid"],
        "cosine_similarity": scores,
    })

    cosine_results[generation] = result

    summary = {
        "generation": generation,
        "cosine_similarity": result["cosine_similarity"].mean(),
        "median_cosine_similarity": result["cosine_similarity"].median(),
    }

    cosine_summaries.append(summary)

    print(pd.DataFrame([summary]).round(4))

# Final combined lexical + semantic table

In [ ]:
final_lexical_semantic = (
    lexical_summary
    .merge(
        bertscore_summary,
        on="generation",
        how="inner"
    )
    .merge(
        pd.DataFrame(cosine_summaries),
        on="generation",
        how="inner"
    )
)

display(
    final_lexical_semantic.round(4)
)

final_lexical_semantic.to_csv(
    EVALUATION_DIR / "lexical_semantic_summary_g0_g3.csv",
    index=False
)

In [ ]:
# Save the cosine-similarity per-sample results
for generation, df in cosine_results.items():
    path = EVALUATION_DIR / f"{generation.lower()}_cosine_similarity.csv"

    df.to_csv(path, index=False)

    print(f"Saved: {path}")

# statistical analysis

# Load the per-sample results

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

EVALUATION_DIR = Path("/content/evaluation")

base = pd.read_csv(
    EVALUATION_DIR / "combined_domain_behavioral_per_sample.csv"
)

print("Base shape:", base.shape)
print("Unique pubids:", base["pubid"].nunique())

# Build one paired table

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

EVALUATION_DIR = Path("/content/evaluation")
generations = ["G0", "G1", "G2", "G3"]

# Start from the domain/behavioral per-sample table
base = pd.read_csv(
    EVALUATION_DIR / "combined_domain_behavioral_per_sample.csv"
)

print("Starting rows:", len(base))

for generation in generations:
    g = generation.lower()

    # --------------------------------------------------
    # 1. Context-support metrics
    # --------------------------------------------------
    context_path = (
        EVALUATION_DIR /
        f"{g}_context_support.csv"
    )

    context_df = pd.read_csv(context_path)

    context_df = context_df[
        [
            "pubid",
            "supported_entity_count",
            "unsupported_entity_count",
            "context_supported_rate",
            "context_unsupported_rate",
        ]
    ].copy()

    context_df = context_df.rename(
        columns={
            "supported_entity_count":
                f"{g}_supported_entity_count",
            "unsupported_entity_count":
                f"{g}_unsupported_entity_count",
            "context_supported_rate":
                f"{g}_context_supported_rate",
            "context_unsupported_rate":
                f"{g}_context_unsupported_rate",
        }
    )

    base = base.merge(
        context_df,
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    # --------------------------------------------------
    # 2. Lexical metrics
    # --------------------------------------------------
    lexical_path = (
        EVALUATION_DIR /
        f"{g}_lexical_metrics_per_row.csv"
    )

    lexical_df = pd.read_csv(lexical_path)

    lexical_columns = [
        "pubid",
        "bleu_1",
        "bleu_2",
        "bleu_3",
        "bleu_4",
        "rouge_l",
        "meteor",
        "chrf_pp",
    ]

    lexical_df = lexical_df[lexical_columns].copy()

    lexical_df = lexical_df.rename(
        columns={
            col: f"{g}_{col}"
            for col in lexical_columns
            if col != "pubid"
        }
    )

    base = base.merge(
        lexical_df,
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    # --------------------------------------------------
    # 3. BERTScore
    # --------------------------------------------------
    bert_path = (
        EVALUATION_DIR /
        f"{g}_bertscore.csv"
    )

    bert_df = pd.read_csv(bert_path)

    bert_columns = [
        "pubid",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1",
    ]

    bert_df = bert_df[bert_columns].copy()

    bert_df = bert_df.rename(
        columns={
            col: f"{g}_{col}"
            for col in bert_columns
            if col != "pubid"
        }
    )

    base = base.merge(
        bert_df,
        on="pubid",
        how="left",
        validate="one_to_one",
    )

    # --------------------------------------------------
    # 4. Cosine similarity
    # --------------------------------------------------
    cosine_path = (
        EVALUATION_DIR /
        f"{g}_cosine_similarity.csv"
    )

    cosine_df = pd.read_csv(cosine_path)

    cosine_df = cosine_df[
        ["pubid", "cosine_similarity"]
    ].copy()

    cosine_df = cosine_df.rename(
        columns={
            "cosine_similarity":
                f"{g}_cosine_similarity"
        }
    )

    base = base.merge(
        cosine_df,
        on="pubid",
        how="left",
        validate="one_to_one",
    )
base.to_csv(EVALUATION_DIR / "combined_lexical_semantic_domain_per_sample.csv", index=False)
print("\nFinal paired table shape:", base.shape)
print("Unique pubids:", base["pubid"].nunique())

In [ ]:
# Validate the paired statistical-analysis table.

generations = ["G0", "G1", "G2", "G3"]

# These should never be missing after a successful merge.
required_count_columns = []

for generation in generations:
    g = generation.lower()

    required_count_columns.extend([
        f"{g}_supported_entity_count",
        f"{g}_unsupported_entity_count",
    ])

missing_count_cols = [
    col for col in required_count_columns
    if col not in base.columns
]

if missing_count_cols:
    print("❌ Missing columns:")
    for col in missing_count_cols:
        print(col)
else:
    print("✅ Context count columns exist.")

# Check whether the merges lost any pubids.
for generation in generations:
    g = generation.lower()

    count_cols = [
        f"{g}_supported_entity_count",
        f"{g}_unsupported_entity_count",
    ]

    missing_rows = base[count_cols].isna().any(axis=1).sum()

    print(
        f"{generation}: {missing_rows} rows missing after context merge"
    )

# Context rates may legitimately be NaN when no entities were detected.
for generation in generations:
    g = generation.lower()

    rate_col = f"{g}_context_supported_rate"

    print(
        f"{generation}: "
        f"{base[rate_col].isna().sum()} NaN context-supported rates "
        f"(allowed when no predicted entities exist)"
    )

assert len(base) == 1000
assert base["pubid"].nunique() == 1000

print("\n✅ 1,000 paired examples confirmed.")

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

generations = ["G0", "G1", "G2", "G3"]

metrics = {
    "Disease F1": "disease_f1",
    "Chemical F1": "chemical_f1",
    "Disease retention": "disease_retention",
    "Chemical retention": "chemical_retention",
    "Context-supported rate": "context_supported_rate",
    "Context-unmatched rate": "context_unsupported_rate",
    "BLEU-1": "bleu_1",
    "BLEU-4": "bleu_4",
    "ROUGE-L": "rouge_l",
    "METEOR": "meteor",
    "chrF++": "chrf_pp",
    "BERTScore F1": "bertscore_f1",
    "Cosine similarity": "cosine_similarity",
    "Average words": "word_count",
    "Average tokens": "token_count",
    "3-gram repetition": "repetition_rate",
    "Perplexity": "perplexity",
}


def get_col(generation, metric_key):
    col = f"{generation.lower()}_{metric_key}"

    if col not in base.columns:
        raise KeyError(f"Missing column: {col}")

    return col


def bootstrap_paired(x, y, n_boot=10000, seed=123):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if n == 0:
        return np.nan, np.nan, np.nan, 0

    differences = y - x
    observed = differences.mean()

    rng = np.random.default_rng(seed)

    indices = rng.integers(
        0,
        n,
        size=(n_boot, n)
    )

    bootstrap_means = differences[indices].mean(axis=1)

    low = np.percentile(bootstrap_means, 2.5)
    high = np.percentile(bootstrap_means, 97.5)

    return observed, low, high, n


def cohens_dz(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    differences = y[mask] - x[mask]

    if len(differences) < 2:
        return np.nan

    sd = np.std(differences, ddof=1)

    if sd == 0:
        return np.nan

    return np.mean(differences) / sd


def relative_change(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    x = x[mask]
    y = y[mask]

    if len(x) == 0:
        return np.nan

    baseline = np.mean(x)
    new = np.mean(y)

    if baseline == 0:
        return np.nan

    return ((new - baseline) / baseline) * 100


comparisons = [
    ("G0", "G1"),
    ("G1", "G2"),
    ("G2", "G3"),
    ("G0", "G3"),
]

rows = []

for metric_name, metric_key in metrics.items():

    for g_start, g_end in comparisons:

        start_col = get_col(g_start, metric_key)
        end_col = get_col(g_end, metric_key)

        x = base[start_col].to_numpy(dtype=float)
        y = base[end_col].to_numpy(dtype=float)

        mean_change, ci_low, ci_high, n = bootstrap_paired(
            x,
            y,
            n_boot=10000,
            seed=123
        )

        effect = cohens_dz(x, y)

        relative = relative_change(x, y)

        mask = np.isfinite(x) & np.isfinite(y)

        if mask.sum() >= 2:

            try:
                _, p_value = wilcoxon(
                    x[mask],
                    y[mask],
                    alternative="two-sided",
                    zero_method="wilcox"
                )
            except ValueError:
                p_value = np.nan

        else:
            p_value = np.nan

        rows.append({
            "metric": metric_name,
            "comparison": f"{g_start} → {g_end}",
            "n": n,
            "mean_change": mean_change,
            "ci_95_low": ci_low,
            "ci_95_high": ci_high,
            "relative_change_percent": relative,
            "cohens_dz": effect,
            "wilcoxon_p": p_value,
        })


stats_df = pd.DataFrame(rows)

display(stats_df.round(5))

In [ ]:
stats_path = (
    EVALUATION_DIR /
    "seed123_paired_statistical_analysis.csv"
)

stats_df.to_csv(stats_path, index=False)

print("Saved:", stats_path)

In [ ]:
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Aggregate paired entity statistics
# ------------------------------------------------------------

comparisons = [
    ("G0", "G1"),
    ("G1", "G2"),
    ("G2", "G3"),
    ("G0", "G3"),
]


def entity_micro_metrics(df, generation):
    """
    Compute micro-level entity precision, recall, and F1
    from aggregate gold/predicted/matched entity counts.
    """
    g = generation.lower()

    gold_disease = df["gold_disease_count"].sum()
    pred_disease = df[f"{g}_disease_count"].sum()
    matched_disease = df[f"{g}_matched_disease_count"].sum()

    gold_chemical = df["gold_chemical_count"].sum()
    pred_chemical = df[f"{g}_chemical_count"].sum()
    matched_chemical = df[f"{g}_matched_chemical_count"].sum()

    disease_precision = (
        matched_disease / pred_disease
        if pred_disease > 0 else np.nan
    )

    disease_recall = (
        matched_disease / gold_disease
        if gold_disease > 0 else np.nan
    )

    disease_f1 = (
        2 * disease_precision * disease_recall /
        (disease_precision + disease_recall)
        if (
            np.isfinite(disease_precision)
            and np.isfinite(disease_recall)
            and disease_precision + disease_recall > 0
        )
        else np.nan
    )

    chemical_precision = (
        matched_chemical / pred_chemical
        if pred_chemical > 0 else np.nan
    )

    chemical_recall = (
        matched_chemical / gold_chemical
        if gold_chemical > 0 else np.nan
    )

    chemical_f1 = (
        2 * chemical_precision * chemical_recall /
        (chemical_precision + chemical_recall)
        if (
            np.isfinite(chemical_precision)
            and np.isfinite(chemical_recall)
            and chemical_precision + chemical_recall > 0
        )
        else np.nan
    )

    return {
        "generation": generation,

        "disease_gold": gold_disease,
        "disease_predicted": pred_disease,
        "disease_matched": matched_disease,
        "disease_precision": disease_precision,
        "disease_recall": disease_recall,
        "disease_f1": disease_f1,

        "chemical_gold": gold_chemical,
        "chemical_predicted": pred_chemical,
        "chemical_matched": matched_chemical,
        "chemical_precision": chemical_precision,
        "chemical_recall": chemical_recall,
        "chemical_f1": chemical_f1,
    }


entity_summary_rows = []

for generation in generations:
    entity_summary_rows.append(
        entity_micro_metrics(base, generation)
    )

entity_summary = pd.DataFrame(entity_summary_rows)

display(entity_summary.round(5))

In [ ]:
entity_path = (
    EVALUATION_DIR /
    "seed123_entity_micro_summary.csv"
)

entity_summary.to_csv(
    entity_path,
    index=False
)

print("Saved:", entity_path)

In [ ]:
import numpy as np
import pandas as pd

def bootstrap_micro_f1_difference(
    df,
    generation_a,
    generation_b,
    entity_type,
    n_boot=10000,
    seed=123,
):
    rng = np.random.default_rng(seed)

    g_a = generation_a.lower()
    g_b = generation_b.lower()

    gold_col = f"gold_{entity_type}_count"
    pred_a_col = f"{g_a}_{entity_type}_count"
    match_a_col = f"{g_a}_matched_{entity_type}_count"
    pred_b_col = f"{g_b}_{entity_type}_count"
    match_b_col = f"{g_b}_matched_{entity_type}_count"

    required = [
        gold_col,
        pred_a_col,
        match_a_col,
        pred_b_col,
        match_b_col,
    ]

    data = df[required].dropna().reset_index(drop=True)

    def micro_f1(sample):
        gold = sample[gold_col].sum()
        pred = sample[pred_a_col].sum() if "A" else None

    def calc_f1(generation_prefix, sample):
        pred_col = (
            f"{generation_prefix.lower()}_{entity_type}_count"
        )
        match_col = (
            f"{generation_prefix.lower()}_matched_{entity_type}_count"
        )

        gold = sample[gold_col].sum()
        pred = sample[pred_col].sum()
        matched = sample[match_col].sum()

        precision = matched / pred if pred > 0 else 0.0
        recall = matched / gold if gold > 0 else 0.0

        if precision + recall == 0:
            return 0.0

        return 2 * precision * recall / (precision + recall)

    observed_a = calc_f1(generation_a, data)
    observed_b = calc_f1(generation_b, data)
    observed_change = observed_b - observed_a

    bootstrap_changes = []

    n = len(data)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = data.iloc[idx]

        f1_a = calc_f1(generation_a, sample)
        f1_b = calc_f1(generation_b, sample)

        bootstrap_changes.append(f1_b - f1_a)

    low = np.percentile(bootstrap_changes, 2.5)
    high = np.percentile(bootstrap_changes, 97.5)

    return {
        "entity_type": entity_type,
        "comparison": f"{generation_a} → {generation_b}",
        "n": n,
        "micro_f1_start": observed_a,
        "micro_f1_end": observed_b,
        "mean_change": observed_change,
        "ci_95_low": low,
        "ci_95_high": high,
    }


rows = []

for entity_type in ["disease", "chemical"]:
    for g_a, g_b in [
        ("G0", "G1"),
        ("G1", "G2"),
        ("G2", "G3"),
        ("G0", "G3"),
    ]:
        rows.append(
            bootstrap_micro_f1_difference(
                base,
                g_a,
                g_b,
                entity_type,
            )
        )

entity_stats = pd.DataFrame(rows)

display(entity_stats.round(5))

In [ ]:
entity_stats_path = (
    EVALUATION_DIR /
    "seed123_entity_micro_bootstrap_statistics.csv"
)

entity_stats.to_csv(
    entity_stats_path,
    index=False
)

print("Saved:", entity_stats_path)